In [1]:
import ast
import json
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
PROJECT_ROOT = Path("..")

PTBXL_DIR = PROJECT_ROOT / "data" / "raw" / "ptb-xl"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv")

scp = pd.read_csv(
    PTBXL_DIR / "scp_statements.csv",
    index_col=0,
)

print("Metadata:", df.shape)
print("SCP statements:", scp.shape)

Metadata: (21799, 28)
SCP statements: (71, 12)


In [4]:
df["scp_codes_dict"] = df["scp_codes"].apply(ast.literal_eval)

In [5]:
df[["ecg_id", "scp_codes", "scp_codes_dict"]].head()

,ecg_id,scp_codes,scp_codes_dict
0,1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}","{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}"
1,2,"{'NORM': 80.0, 'SBRAD': 0.0}","{'NORM': 80.0, 'SBRAD': 0.0}"
2,3,"{'NORM': 100.0, 'SR': 0.0}","{'NORM': 100.0, 'SR': 0.0}"
3,4,"{'NORM': 100.0, 'SR': 0.0}","{'NORM': 100.0, 'SR': 0.0}"
4,5,"{'NORM': 100.0, 'SR': 0.0}","{'NORM': 100.0, 'SR': 0.0}"


In [6]:
diagnostic_mapping = scp[scp["diagnostic"] == 1]["diagnostic_class"].dropna().to_dict()

print("Number of diagnostic SCP codes:", len(diagnostic_mapping))

Number of diagnostic SCP codes: 44


In [7]:
def get_diagnostic_classes(codes, mapping):
    classes = set()

    for code in codes:

        diagnostic_class = mapping.get(code)

        if pd.notna(diagnostic_class):
            classes.add(diagnostic_class)

    return sorted(classes)

In [8]:
df["diagnostic_classes"] = df["scp_codes_dict"].apply(
    lambda codes: get_diagnostic_classes(codes, diagnostic_mapping)
)

In [9]:
df[["ecg_id", "diagnostic_classes"]].head(20)

,ecg_id,diagnostic_classes
0,1,[NORM]
1,2,[NORM]
2,3,[NORM]
3,4,[NORM]
4,5,[NORM]
5,6,[NORM]
6,7,[NORM]
7,8,[MI]
8,9,[NORM]
9,10,[NORM]


In [10]:
no_diagnostic = df["diagnostic_classes"].apply(len).eq(0)

print("Without diagnostic class:", no_diagnostic.sum())

print("With diagnostic class:", (~no_diagnostic).sum())

Without diagnostic class: 411
With diagnostic class: 21388


In [11]:
class_counts = {}

for classes in df["diagnostic_classes"]:

    for cls in classes:

        class_counts[cls] = class_counts.get(cls, 0) + 1

class_counts = pd.Series(class_counts).sort_values(ascending=False)

class_counts

NORM    9514
MI      5469
STTC    5235
CD      4898
HYP     2649
dtype: int64

In [12]:
available_classes = class_counts.index.tolist()

print("Diagnostic classes:")

for cls in available_classes:
    print(f"- {cls}: " f"{class_counts[cls]}")

Diagnostic classes:
- NORM: 9514
- MI: 5469
- STTC: 5235
- CD: 4898
- HYP: 2649


In [13]:
class_definitions = (
    scp[scp["diagnostic"] == 1][["diagnostic_class", "description"]]
    .drop_duplicates()
    .sort_values("diagnostic_class")
)

class_definitions

,diagnostic_class,description
LAFB,CD,left anterior fascicular block
IRBBB,CD,incomplete right bundle branch block
1AVB,CD,first degree AV block
IVCD,CD,non-specific intraventricular conduction distu...
CRBBB,CD,complete right bundle branch block
CLBBB,CD,complete left bundle branch block
LPFB,CD,left posterior fascicular block
WPW,CD,Wolf-Parkinson-White syndrome
ILBBB,CD,incomplete left bundle branch block
3AVB,CD,third degree AV block


In [14]:
fold_counts = df["strat_fold"].value_counts().sort_index()

fold_counts

strat_fold
1     2175
2     2181
3     2192
4     2174
5     2174
6     2173
7     2176
8     2173
9     2183
10    2198
Name: count, dtype: int64

In [15]:
train_df = df[df["strat_fold"].between(1, 8)].copy()

val_df = df[df["strat_fold"] == 9].copy()

test_df = df[df["strat_fold"] == 10].copy()

In [16]:
print("Train:", train_df.shape)

print("Validation:", val_df.shape)

print("Test:", test_df.shape)

Train: (17418, 30)
Validation: (2183, 30)
Test: (2198, 30)


In [17]:
train_patients = set(train_df["patient_id"])

val_patients = set(val_df["patient_id"])

test_patients = set(test_df["patient_id"])

In [18]:
print("Train ∩ Validation:", len(train_patients & val_patients))

print("Train ∩ Test:", len(train_patients & test_patients))

print("Validation ∩ Test:", len(val_patients & test_patients))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [19]:
for fold_name, subset in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:

    counts = {}

    for classes in subset["diagnostic_classes"]:

        for cls in classes:

            counts[cls] = counts.get(cls, 0) + 1

    print(f"\n{fold_name.upper()}")

    print(pd.Series(counts).sort_values(ascending=False))


TRAIN
NORM    7596
MI      4379
STTC    4186
CD      3907
HYP     2119
dtype: int64

VALIDATION
NORM    955
MI      540
STTC    528
CD      495
HYP     268
dtype: int64

TEST
NORM    963
MI      550
STTC    521
CD      496
HYP     262
dtype: int64


In [20]:
train_df.to_csv(PROCESSED_DIR / "ptbxl_train_metadata.csv", index=False)

val_df.to_csv(PROCESSED_DIR / "ptbxl_val_metadata.csv", index=False)

test_df.to_csv(PROCESSED_DIR / "ptbxl_test_metadata.csv", index=False)

In [21]:
class_counts.to_csv(REPORTS_DIR / "ptbxl_diagnostic_class_counts.csv", header=["count"])

In [22]:
class_definitions.to_csv(
    REPORTS_DIR / "ptbxl_diagnostic_class_definitions.csv", index=False
)

In [23]:
summary = {
    "total_recordings": int(len(df)),
    "total_patients": int(df["patient_id"].nunique()),
    "train_recordings": int(len(train_df)),
    "validation_recordings": int(len(val_df)),
    "test_recordings": int(len(test_df)),
    "diagnostic_classes": (available_classes),
}

In [24]:
with open(REPORTS_DIR / "ptbxl_label_preparation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)